### Dogs

In [8]:
import json
import math
import os
import re
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from resize_image import resize_image
from tqdm.std import tqdm  # Using standard tqdm to avoid ipywidgets warning


def _strip_jsonc_comments(text: str) -> str:
    # VS Code settings.json allows comments and trailing commas.
    output = []
    in_string = False
    escaped = False
    line_comment = False
    block_comment = False
    i = 0
    while i < len(text):
        char = text[i]
        next_char = text[i + 1] if i + 1 < len(text) else ""
        if line_comment:
            if char == "\n":
                line_comment = False
                output.append(char)
            i += 1
            continue
        if block_comment:
            if char == "*" and next_char == "/":
                block_comment = False
                i += 2
            else:
                i += 1
            continue
        if in_string:
            output.append(char)
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            i += 1
            continue
        if char == '"':
            in_string = True
            output.append(char)
        elif char == "/" and next_char == "/":
            line_comment = True
            i += 2
            continue
        elif char == "/" and next_char == "*":
            block_comment = True
            i += 2
            continue
        else:
            output.append(char)
        i += 1
    return re.sub(r",\s*([}\]])", r"\1", "".join(output))


def get_hf_token() -> str:
    home = Path.home()
    settings_paths = [
        home / "Library/Application Support/Code/User/settings.json",
        home / "Library/Application Support/Code - Insiders/User/settings.json",
        home / ".config/Code/User/settings.json",
        home / ".config/Code - Insiders/User/settings.json",
        home / "AppData/Roaming/Code/User/settings.json",
        home / "AppData/Roaming/Code - Insiders/User/settings.json",
    ]
    custom_settings_path = os.getenv("VSCODE_SETTINGS_PATH")
    if custom_settings_path:
        settings_paths.insert(0, Path(custom_settings_path).expanduser())

    for settings_path in settings_paths:
        if not settings_path.is_file():
            continue
        try:
            settings = json.loads(_strip_jsonc_comments(settings_path.read_text()))
        except (OSError, json.JSONDecodeError):
            continue
        for key in ("HF_TOKEN", "hf_token", "huggingface.token"):
            token = settings.get(key)
            if isinstance(token, str) and token.strip():
                return token.strip()

    token = os.getenv("HF_TOKEN")
    if token:
        return token.strip()
    raise ValueError("Add HF_TOKEN to VS Code user settings.json or set the HF_TOKEN environment variable.")



In [ ]:

TARGET_BREEDS = {
    "German shepherd", "Golden retriever", "American Staffordshire terrier",
    "Appenzeller", "Basenji", "Beagle", "Basset",
    "Bernese mountain dog", "Blenheim spaniel", "Border collie",
    "Boxer", "Brittany spaniel", "Cardigan", "Chihuahua",
    "Clumber", "Cocker spaniel", "Curly-coated retriever",
    "English foxhound", "English springer", "EntleBucher",
    "Eskimo dog", "Great Pyrenees", "Greater Swiss Mountain dog",
    "Japanese spaniel", "Kuvasz", "Labrador retriever",
    "Leonberg", "Malamute", "Malinois", "Miniature pinscher",
    "Newfoundland", "Norfolk terrier", "Redbone",
    "Rhodesian ridgeback", "Rottweiler", "Saint Bernard",
    "Samoyed", "Siberian husky", "Sussex spaniel", "Vizsla",
    "Walker hound", "Welsh springer spaniel",
}


def normalize_breed_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


def fetch_selected_dogs_sharded(
    output_dir: str = "images/dogs",
    total_target: int = 10000,
    hf_token: str | None = None,  # Optional: "hf_xxxxxxxx..."
):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    max_per_class = math.ceil(total_target / len(TARGET_BREEDS))

    target_breeds = {normalize_breed_name(breed): breed for breed in TARGET_BREEDS}
    class_counts = {breed: 0 for breed in target_breeds}
    for existing_file in out_path.glob("*.jpg"):
        existing_name = existing_file.stem.rsplit("_", 2)[0].replace("_", " ")
        existing_key = normalize_breed_name(existing_name)
        if existing_key in class_counts:
            class_counts[existing_key] += 1

    total_saved = sum(class_counts.values())
    print(f"Found {total_saved} existing selected-breed images; downloading only the remainder.")
    if total_saved >= total_target:
        print(f"Already have at least {total_target} images in '{out_path.resolve()}'.")
        return

    pbar = tqdm(total=total_target, initial=total_saved, desc="Downloading Dog Images")
    labels = None

    # ImageNet 128x128 currently has 13 training shards (train-00000 to train-00012).
    num_train_shards = 13
    for shard_idx in range(num_train_shards):
        if total_saved >= total_target:
            break

        shard_file = f"data/train-{shard_idx:05d}-of-{num_train_shards:05d}.parquet"
        print(f"\n[Shard {shard_idx + 1}/{num_train_shards}] Downloading '{shard_file}'...")

        # Download the shard explicitly, then read it as a local parquet file.
        # This avoids the Hub dataset builder's multi-split inference error.
        shard_path = hf_hub_download(
            repo_id="benjamin-paine/imagenet-1k-128x128",
            filename=shard_file,
            repo_type="dataset",
            token=hf_token,
        )
        shard_ds = load_dataset(
            "parquet",
            data_files={"train": shard_path},
            split="train",
        )

        if labels is None and "label" in shard_ds.features:
            labels = shard_ds.features["label"].names

        for sample in shard_ds:
            if total_saved >= total_target:
                break

            label_idx = sample["label"]
            raw_breed_name = labels[label_idx].split(",")[0].strip()
            breed_key = normalize_breed_name(raw_breed_name)

            if breed_key in class_counts and class_counts[breed_key] < max_per_class:
                image = sample["image"]

                image = resize_image(image, (64, 64))
                breed_name = target_breeds[breed_key]
                safe_breed_name = breed_name.replace(" ", "_")
                filename = f"{safe_breed_name}_{class_counts[breed_key]:03d}_{total_saved:05d}.jpg"
                image.save(out_path / filename, "JPEG", quality=95)

                class_counts[breed_key] += 1
                total_saved += 1
                pbar.update(1)

    pbar.close()
    print(
        f"\nDone! Successfully saved {total_saved} dog images into '{out_path.resolve()}'."
    )
    print("Per-breed counts:", {target_breeds[key]: count for key, count in class_counts.items()})


if __name__ == "__main__":
    fetch_selected_dogs_sharded(
        output_dir="images/dogs",
        total_target=10000,
        hf_token=get_hf_token(),
    )

# Car

In [ ]:
import os
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from resize_image import resize_image
from tqdm.std import tqdm


def fetch_random_cars(
    output_dir: str = "images/car",
    total_target: int = 10000,
    seed: int = 42,
    hf_token: str | None = None,
    max_workers: int = 128,
):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # List the image paths, sample only what is needed, and download those
    # paths concurrently. This avoids preparing the complete dataset first.
    token = hf_token or get_hf_token()
    if not token:
        raise ValueError("Set HF_TOKEN or pass hf_token to authenticate with Hugging Face.")
    repo_id = "pawlo2013/Cars196"
    all_files = HfApi(token=token).list_repo_files(repo_id, repo_type="dataset")
    image_files = sorted(
        path for path in all_files if path.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    if len(image_files) < total_target:
        raise ValueError(f"Dataset has only {len(image_files)} images; need {total_target}.")

    selected_files = random.Random(seed).sample(image_files, total_target)

    # Regeneration is deterministic for the same seed and leaves exactly 10,000 files.
    for existing_file in out_path.glob("*.jpg"):
        existing_file.unlink()

    def download_and_save(item):
        output_index, repo_file = item
        local_file = hf_hub_download(
            repo_id=repo_id, filename=repo_file, repo_type="dataset", token=token
        )
        with Image.open(local_file) as image:
            resized = resize_image(image, (64, 64))
            resized.save(out_path / f"car_{output_index:05d}.jpg", "JPEG", quality=95)

    items = enumerate(selected_files)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(download_and_save, item) for item in items]
        for future in tqdm(
            as_completed(futures), total=total_target, desc="Downloading and saving Car Images"
        ):
            future.result()

    print(f"Saved {total_target} distinct 64x64 car images to '{out_path.resolve()}'.")


fetch_random_cars(hf_token=get_hf_token())


README.md:   0%|          | 0.00/434 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/8144 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/8041 [00:00<?, ?it/s]

car_data/car_data/train/AM General Humme(…): reconstructing file:   0%|          |  0.00B / 3.39kB            

car_data/car_data/train/AM General Humme(…): reconstructing file:   0%|          |  0.00B / 1.99kB            

...

car_data/car_data/train/Isuzu Ascender S(…): downloading bytes:           |  0.00B            

car_data/car_data/train/Isuzu Ascender S(…): reconstructing file:   0%|          |  0.00B / 10.2kB            

car_data/car_data/train/Isuzu Ascender S(…): downloading bytes:           |  0.00B            

car_data/car_data/train/Isuzu Ascender S(…): reconstructing file:   0%|          |  0.00B / 18.8kB            

car_data/car_data/train/Isuzu Ascender S(…): downloading bytes:           |  0.00B            

### Birds

In [11]:
import math
import os
import re
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from resize_image import resize_image
from tqdm.std import tqdm


TARGET_BIRDS = {
    "brambling", "goldfinch", "house finch", "junco",
    "indigo bunting", "robin", "bulbul", "jay",
    "magpie", "chickadee", "water ouzel", "kite",
    "bald eagle", "vulture", "great grey owl", "African grey",
    "macaw", "sulphur-crested cockatoo", "lorikeet", "coucal",
    "bee eater", "hornbill", "hummingbird", "jacamar", "toucan",
}


def normalize_bird_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


def fetch_selected_birds_sharded(
    output_dir: str = "images/birds",
    total_target: int = 10000,
    hf_token: str | None = None,
):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    max_per_class = math.ceil(total_target / len(TARGET_BIRDS))
    target_birds = {normalize_bird_name(bird): bird for bird in TARGET_BIRDS}
    class_counts = {bird: 0 for bird in target_birds}

    for existing_file in out_path.glob("*.jpg"):
        existing_name = existing_file.stem.rsplit("_", 2)[0].replace("_", " ")
        existing_key = normalize_bird_name(existing_name)
        if existing_key in class_counts:
            class_counts[existing_key] += 1

    total_saved = sum(class_counts.values())
    print(f"Found {total_saved} existing selected-bird images; downloading only the remainder.")
    if total_saved >= total_target:
        print(f"Already have at least {total_target} images in '{out_path.resolve()}'.")
        return

    pbar = tqdm(total=total_target, initial=total_saved, desc="Downloading Bird Images")
    labels = None
    num_train_shards = 13

    for shard_idx in range(num_train_shards):
        if total_saved >= total_target:
            break

        shard_file = f"data/train-{shard_idx:05d}-of-{num_train_shards:05d}.parquet"
        print(f"\n[Shard {shard_idx + 1}/{num_train_shards}] Downloading '{shard_file}'...")
        shard_path = hf_hub_download(
            repo_id="benjamin-paine/imagenet-1k-128x128",
            filename=shard_file,
            repo_type="dataset",
            token=hf_token,
        )
        shard_ds = load_dataset("parquet", data_files={"train": shard_path}, split="train")

        if labels is None and "label" in shard_ds.features:
            labels = shard_ds.features["label"].names

        for sample in shard_ds:
            if total_saved >= total_target:
                break

            label_idx = sample["label"]
            raw_bird_name = labels[label_idx].split(",")[0].strip()
            bird_key = normalize_bird_name(raw_bird_name)

            if bird_key in class_counts and class_counts[bird_key] < max_per_class:
                image = resize_image(sample["image"], (64, 64))
                bird_name = target_birds[bird_key]
                safe_bird_name = bird_name.replace(" ", "_").replace("-", "_")
                filename = f"{safe_bird_name}_{class_counts[bird_key]:03d}_{total_saved:05d}.jpg"
                image.save(out_path / filename, "JPEG", quality=95)

                class_counts[bird_key] += 1
                total_saved += 1
                pbar.update(1)

    pbar.close()
    print(f"\nDone! Successfully saved {total_saved} bird images into '{out_path.resolve()}'.")
    print("Per-bird counts:", {target_birds[key]: count for key, count in class_counts.items()})


fetch_selected_birds_sharded(
    output_dir="images/birds",
    total_target=10000,
    hf_token=get_hf_token(),
)


Found 0 existing selected-bird images; downloading only the remainder.



[Shard 1/13] Downloading 'data/train-00000-of-00013.parquet'...



[Shard 2/13] Downloading 'data/train-00001-of-00013.parquet'...



Done! Successfully saved 10000 bird images into '/Users/tanishsahu/Desktop/deep-learning/Mitigating-biasness-in-generative-models/images/birds'.
Per-bird counts: {'jacamar': 400, 'great grey owl': 400, 'sulphur-crested cockatoo': 400, 'vulture': 400, 'macaw': 400, 'goldfinch': 400, 'hummingbird': 400, 'African grey': 400, 'bulbul': 400, 'indigo bunting': 400, 'junco': 400, 'bald eagle': 400, 'lorikeet': 400, 'jay': 400, 'house finch': 400, 'kite': 400, 'brambling': 400, 'bee eater': 400, 'robin': 400, 'magpie': 400, 'chickadee': 400, 'hornbill': 400, 'coucal': 400, 'water ouzel': 400, 'toucan': 400}


### ButterFly

In [ ]:
import csv
import math
import random
import re
from pathlib import Path

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from PIL import Image
from resize_image import resize_image
from tqdm.std import tqdm


# ImageNet classes 321-326 (the six butterfly classes in the standard
# 1-based ImageNet class list): admiral, ringlet, monarch, cabbage
# butterfly, sulphur butterfly, and lycaenid. Matching names instead of
# numeric label IDs avoids a 0-based/1-based indexing mistake.
IMAGENET_BUTTERFLY_LABELS = {
    "admiral",
    "ringlet",
    "monarch",
    "cabbage butterfly",
    "sulphur butterfly",
    "lycaenid",
}


def normalize_butterfly_label(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


def _safe_butterfly_label(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _find_kaggle_file(root: Path, filename: str) -> Path:
    matches = [path for path in root.rglob("*") if path.name.lower() == filename.lower()]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} below {root}")
    return matches[0]


def fetch_butterflies(
    output_dir: str = "images/butterfly",
    total_target: int = 10000,
    seed: int = 42,
    hf_token: str | None = None,
):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # Regenerate this mixed dataset deterministically.
    for existing_file in out_path.glob("*.jpg"):
        existing_file.unlink()

    target_labels = {
        normalize_butterfly_label(label): label for label in IMAGENET_BUTTERFLY_LABELS
    }
    imagenet_counts = {label: 0 for label in target_labels}
    total_saved = 0
    labels = None
    num_train_shards = 13

    imagenet_pbar = tqdm(desc="Saving ImageNet butterfly images")
    for shard_idx in range(num_train_shards):
        shard_file = f"data/train-{shard_idx:05d}-of-{num_train_shards:05d}.parquet"
        print(f"\n[ImageNet shard {shard_idx + 1}/{num_train_shards}] {shard_file}")
        shard_path = hf_hub_download(
            repo_id="benjamin-paine/imagenet-1k-128x128",
            filename=shard_file,
            repo_type="dataset",
            token=hf_token,
        )
        shard_ds = load_dataset("parquet", data_files={"train": shard_path}, split="train")
        if labels is None:
            labels = shard_ds.features["label"].names

        for sample in shard_ds:
            raw_label = labels[sample["label"]].split(",")[0].strip()
            label_key = normalize_butterfly_label(raw_label)
            if label_key not in target_labels:
                continue

            image = resize_image(sample["image"], (64, 64))
            label_name = target_labels[label_key]
            filename = (
                f"imagenet_{_safe_butterfly_label(label_name)}_"
                f"{imagenet_counts[label_key]:04d}_{total_saved:05d}.jpg"
            )
            image.save(out_path / filename, "JPEG", quality=95)
            imagenet_counts[label_key] += 1
            total_saved += 1
            imagenet_pbar.update(1)

    imagenet_pbar.close()
    print("ImageNet counts:", {target_labels[key]: count for key, count in imagenet_counts.items()})
    if total_saved > total_target:
        raise ValueError(
            f"The six ImageNet classes produced {total_saved} images,"
            f" which exceeds the target of {total_target}."
        )

    remaining = total_target - total_saved
    if remaining:
        try:
            import kagglehub
        except ImportError as exc:
            raise ImportError("Install KaggleHub first: pip install kagglehub") from exc

        kaggle_root = Path(
            kagglehub.dataset_download("phucthaiv02/butterfly-image-classification")
        )
        training_csv = _find_kaggle_file(kaggle_root, "Training_set.csv")
        train_dirs = [
            path for path in kaggle_root.rglob("*")
            if path.is_dir() and path.name.lower() == "train"
        ]
        if not train_dirs:
            raise FileNotFoundError(f"Could not find the Kaggle train directory below {kaggle_root}")
        train_dir = train_dirs[0]

        with training_csv.open(newline="", encoding="utf-8") as csv_file:
            reader = csv.DictReader(csv_file)
            fieldnames = reader.fieldnames or []
            filename_field = next(
                (field for field in fieldnames if field.lower() in {"filename", "file_name", "image"}),
                fieldnames[0],
            )
            label_field = next(
                (field for field in fieldnames if field.lower() in {"label", "class", "category"}),
                fieldnames[-1],
            )
            kaggle_records = []
            for row in reader:
                image_path = train_dir / row[filename_field]
                if image_path.is_file():
                    kaggle_records.append((image_path, row[label_field]))

        if len(kaggle_records) < remaining:
            raise ValueError(
                f"Kaggle has {len(kaggle_records)} usable training images;"
                f" need {remaining} more."
            )

        selected_kaggle = random.Random(seed).sample(kaggle_records, remaining)
        for kaggle_index, (image_path, label) in enumerate(
            tqdm(selected_kaggle, desc="Saving Kaggle butterfly images")
        ):
            with Image.open(image_path) as image:
                resized = resize_image(image, (64, 64))
                filename = (
                    f"kaggle_{_safe_butterfly_label(label)}_"
                    f"{kaggle_index:05d}_{total_saved:05d}.jpg"
                )
                resized.save(out_path / filename, "JPEG", quality=95)
            total_saved += 1

    if total_saved != total_target:
        raise RuntimeError(f"Saved {total_saved} images; expected {total_target}.")
    print(f"Done! Saved {total_saved} 64x64 butterfly images to '{out_path.resolve()}'.")


fetch_butterflies(
    output_dir="images/butterfly",
    total_target=10000,
    hf_token=get_hf_token(),
)


### Leaf

In [12]:
import random
import re
from collections import defaultdict
from pathlib import Path

from PIL import Image
from resize_image import resize_image
from tqdm.std import tqdm


def _safe_leaf_label(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def fetch_leafsnap_processed(
    output_dir: str = "images/leaf",
    total_target: int = 10000,
    seed: int = 42,
    kaggle_dataset: str = "vandat2601/leafsnap-processed",
):
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError("Install KaggleHub first: pip install kagglehub") from exc

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    dataset_root = Path(kagglehub.dataset_download(kaggle_dataset))
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # The processed Kaggle release is organized by species directories.
    # Include both train and test trees when present, but never the original
    # Leafsnap source dataset or any external fallback.
    split_dirs = [
        path for path in dataset_root.rglob("*")
        if path.is_dir() and path.name.lower() in {"train", "test"}
    ]
    records = []
    seen_paths = set()
    search_roots = split_dirs or [dataset_root]
    for search_root in search_roots:
        for image_path in search_root.rglob("*"):
            if not image_path.is_file() or image_path.suffix.lower() not in image_extensions:
                continue
            resolved_path = image_path.resolve()
            if resolved_path in seen_paths:
                continue
            seen_paths.add(resolved_path)
            relative_parts = image_path.relative_to(search_root).parts
            label = relative_parts[0] if len(relative_parts) > 1 else search_root.name
            records.append((image_path, label))

    if len(records) < total_target:
        raise ValueError(
            f"Leafsnap Processed contains {len(records)} usable images;"
            f" need {total_target}. No other dataset was used."
        )

    # Deterministic sample across all processed Leafsnap species.
    selected_records = random.Random(seed).sample(records, total_target)
    for existing_file in out_path.glob("*.jpg"):
        existing_file.unlink()

    label_counts = defaultdict(int)
    for output_index, (image_path, label) in enumerate(
        tqdm(selected_records, total=total_target, desc="Saving Leafsnap images")
    ):
        with Image.open(image_path) as image:
            resized = resize_image(image, (64, 64))
            safe_label = _safe_leaf_label(label)
            filename = f"leaf_{safe_label}_{label_counts[label]:04d}_{output_index:05d}.jpg"
            resized.save(out_path / filename, "JPEG", quality=95)
        label_counts[label] += 1

    print(f"Saved {total_target} 64x64 Leafsnap images to '{out_path.resolve()}'.")
    print(f"Source images found: {len(records)}; classes represented: {len(label_counts)}")


fetch_leafsnap_processed(
    output_dir="images/leaf",
    total_target=10000,
    seed=42,
)


100%|██████████| 137M/137M [01:09<00:00, 2.07MB/s] 

Extracting files...



Saving Leafsnap images: 100%|██████████| 10000/10000 [00:07<00:00, 1333.10it/s]

Saved 10000 64x64 Leafsnap images to '/Users/tanishsahu/Desktop/deep-learning/Mitigating-biasness-in-generative-models/images/leaf'.
Source images found: 24624; classes represented: 185
